In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

amazon = spark.table(
    "workspace.amazon_data.amazon"
).toPandas()

amazon["OrderDate"] = pd.to_datetime(
    amazon["OrderDate"]
)

### Create a prediction cutoff

In [0]:
max_date = amazon["OrderDate"].max()

cutoff_date = max_date - pd.Timedelta(days=90)

print("Maximum date:", max_date)
print("Cutoff date:", cutoff_date)

Maximum date: 2024-12-29 00:00:00
Cutoff date: 2024-09-30 00:00:00


### Create historical customer features

In [0]:
history = amazon[
    amazon["OrderDate"] <= cutoff_date
].copy()

### Create features

In [0]:
reference_date = cutoff_date

features = (
    history.groupby("CustomerID")
    .agg(
        Recency=("OrderDate", lambda x:
                 (reference_date - x.max()).days),

        Frequency=("OrderID", "nunique"),

        Monetary=("TotalAmount", "sum"),

        AverageOrderValue=("TotalAmount", "mean"),

        TotalQuantity=("Quantity", "sum"),

        UniqueProducts=("ProductID", "nunique"),

        UniqueCategories=("Category", "nunique")
    )
    .reset_index()
)

In [0]:
features.head()

,CustomerID,Recency,Frequency,Monetary,AverageOrderValue,TotalQuantity,UniqueProducts,UniqueCategories
0,CUST000001,66,1,446.48,446.480000,1,1,1
1,CUST000002,972,3,1389.93,463.310000,9,3,2
2,CUST000003,1372,1,1116.91,1116.910000,5,1,1
3,CUST000004,105,3,2660.06,886.686667,7,3,2
4,CUST000005,549,1,1089.56,1089.560000,4,1,1


Create churn labels

In [0]:
future = amazon[
    amazon["OrderDate"] > cutoff_date
].copy()

In [0]:
future_purchases = (
    future.groupby("CustomerID")["OrderDate"]
    .min()
    .reset_index()
)

future_purchases = future_purchases.rename(
    columns={
        "OrderDate": "FuturePurchaseDate"
    }
)

### Create churn target

In [0]:
features = features.merge(
    future_purchases,
    on="CustomerID",
    how="left"
)

features["Churn"] = np.where(
    features["FuturePurchaseDate"].isna(),
    1,
    0
)

In [0]:
features["Churn"].value_counts()

Churn
1    38625
0     3919
Name: count, dtype: int64

In [0]:
features["Churn"].value_counts(
    normalize=True
).mul(100).round(2)

Churn
1    90.79
0     9.21
Name: proportion, dtype: float64

### Select ML features

In [0]:
ml_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "UniqueProducts",
    "UniqueCategories"
]

X = features[ml_features]
y = features["Churn"]

rain/test split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Logistic **Regression**

In [0]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])

logistic_model.fit(
    X_train,
    y_train
)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [0]:
logistic_pred = logistic_model.predict(X_test)

logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

### Random Forest

In [0]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(
    X_train,
    y_train
)

rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]

### Model comparison

In [0]:
def evaluate_model(name, y_true, predictions, probabilities):

    return {
        "Model": name,
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probabilities
        )
    }

In [0]:
model_results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_pred,
        logistic_prob
    ),

    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred,
        rf_prob
    )
])

model_results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.907862,0.907862,1.000000,0.951706,0.512882
1,Random Forest,0.755318,0.908380,0.812427,0.857729,0.498775


### Customer Risk Scoring

In [0]:
final_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

final_model.fit(
    X,
    y
)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### ## Generate churn probability

In [0]:
features["ChurnProbability"] = (
    final_model.predict_proba(X)[:, 1]
)

In [0]:
features["RiskLevel"] = pd.cut(
    features["ChurnProbability"],
    bins=[0, 0.3, 0.7, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

In [0]:
customer_risk = features[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "AverageOrderValue",
        "UniqueProducts",
        "UniqueCategories",
        "ChurnProbability",
        "RiskLevel"
    ]
].copy()

In [0]:
customer_risk.sort_values(
    "ChurnProbability",
    ascending=False
).head(20)

,CustomerID,Recency,Frequency,Monetary,AverageOrderValue,UniqueProducts,UniqueCategories,ChurnProbability,RiskLevel
36360,CUST042689,800,8,10726.80,1340.850000,7,5,0.932574,High
27273,CUST032008,229,7,9514.36,1359.194286,7,5,0.923807,High
19259,CUST022597,233,8,8886.63,1110.828750,8,5,0.918795,High
5154,CUST006075,398,8,7824.24,978.030000,8,6,0.913145,High
41470,CUST048737,283,9,10125.53,1125.058889,8,5,0.912424,High
997,CUST001187,141,7,10034.75,1433.535714,6,5,0.906611,High
36688,CUST043080,536,9,9244.92,1027.213333,9,5,0.905900,High
3823,CUST004513,0,7,7963.36,1137.622857,7,5,0.904727,High
40749,CUST047880,260,7,10186.18,1455.168571,6,5,0.901846,High
6524,CUST007695,402,7,9928.54,1418.362857,7,4,0.896828,High


In [0]:
spark.createDataFrame(
    customer_risk
).write.mode(
    "overwrite"
).saveAsTable(
    "amazon_customer_risk"
)